In [1]:
pip install -U google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 13.1 MB/s  0:00:00
  Attempting uninstall: google-auth
    Found existing installation: google-auth 2.49.0
    Uninstalling google-auth-2.49.0:
      Successfully uninstalled google-auth-2.49.0
  Attempting uninstall: google-genai
    Found existing installation: google-genai 2.12.1
    Uninstalling google-genai-2.12.1:
      Successfully uninstalled google-genai-2.12.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [google-genai]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.51.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-cloud-aiplatform 2.7.0 requires google-genai<3.0.0,>=1.66.0; python_version >= "3.10", but you have google-genai 3.5.0 which is incompatible.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.2 which is

In [2]:


import os
from kaggle_secrets import UserSecretsClient
from google import genai
from google.genai import types

# Secrets se key padho aur environment variable me set karo
os.environ["GEMINI_API_KEY"] = UserSecretsClient().get_secret("GEMINI_API_KEY")
client = genai.Client()

MODEL = "gemini-3.5-flash"  # error aaye to quickstart page se current model naam copy karo

# Pehla call
response = client.models.generate_content(
    model=MODEL,
    contents="Explain how AI works in a few words",
)
print("PEHLA CALL:\n", response.text, "\n")

# Experiment 1: same sawaal, temperature 0 vs 1.5, 3-3 baar
question = "Ek chhoti si kahani ki pehli line likho."
for temp in (0, 1.5):
    print(f"--- temperature = {temp} ---")
    for i in range(3):
        r = client.models.generate_content(
            model=MODEL,
            contents=question,
            config=types.GenerateContentConfig(temperature=temp),
        )
        print(i + 1, r.text.strip())

# Experiment 2: system prompt badal ke dekho
for persona in ("Tum ek strict teacher ho.", "Tum ek dost ho jo mazaak me samjhata hai."):
    r = client.models.generate_content(
        model=MODEL,
        contents="Token kya hota hai, LLM ke context me?",
        config=types.GenerateContentConfig(system_instruction=persona),
    )
    print(f"\n[{persona}]\n", r.text)

# Experiment 3: Hindi me poocho
r = client.models.generate_content(
    model=MODEL,
    contents="Context window kya hoti hai? Hindi me 3 line me batao.",
)
print("\nHINDI:\n", r.text)

PEHLA CALL:
 **AI learns patterns from vast amounts of data to make predictions, solve problems, and generate answers.**

In short: 
1. **Inputs** data 
2. **Finds** patterns 
3. **Predicts** outcomes 

--- temperature = 0 ---
1 **"Baarish ki pehli boond girte hi uska purana phone baj utha—ek aise number se, jo saalon pehle band ho chuka tha."** 

*(बारिश की पहली बूंद गिरते ही उसका पुराना फोन बज उठा—एक ऐसे नंबर से, जो सालों पहले बंद हो चुका था।)*
2 "Dadi ki purani ghadi pichhle dus saalon se band thi, lekin aaj raat thik baraah baje wo achanak chalne lagi."
3 **"Gaon ke us purane makaan ki khidki mein, aaj kayi saalon baad pehli baar roshni dikhi thi."**

*(गाँव के उस पुराने मकान की खिड़की में, आज कई सालों बाद पहली बार रोशनी दिखी थी।)*
--- temperature = 1.5 ---


ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}